# Customer Churn Prediction
## Notebook 2 of 8 — Data Cleaning

This project predicts which telecom customers are likely to **churn** (cancel their service) so the business can reach them with retention offers *before* they leave. It walks through the full data-science lifecycle — exploration, cleaning, EDA, feature engineering, modelling, evaluation, and a tuned final model.

**Business problem:** Winning a new customer costs far more than keeping an existing one. This telecom loses roughly **27% of its customers**, and the leadership team wants a reliable, data-driven way to flag at-risk customers early enough to act.

**Tools & techniques:** Python · pandas · NumPy · Matplotlib · Seaborn · scikit-learn · XGBoost · SMOTE (imbalanced-learn) · joblib

> **This notebook:** the single source of truth for cleaning. The cleaning logic lives in `src/data_preprocessing.clean_data()` — a reusable, unit-tested function we import here, run, and save the result of.

**Author:** La Yaung Linn Lett  &nbsp;·&nbsp;  **Last updated:** June 2026

---

## 1. Load the raw data

**What:** Read the original dataset, untouched, and import our preprocessing helpers.

**Why:** Cleaning always starts from the raw source so the steps are fully reproducible. Rather than writing the cleaning logic inline, we import it from `src/data_preprocessing.py` — the same tested function is reused across the project (and validated in `tests/`).

In [1]:
# Standard library
import sys
import warnings
from pathlib import Path

# Third-party
import pandas as pd

# Local — reusable, unit-tested preprocessing functions (see src/data_preprocessing.py)
sys.path.append(str(Path.cwd().parent))
from src.data_preprocessing import load_raw_data, clean_data

warnings.filterwarnings("ignore", category=UserWarning)

df_raw = load_raw_data("../data/raw/telco-customer-churn.csv")
print(f"Loaded raw data: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print("TotalCharges dtype before cleaning:", df_raw["TotalCharges"].dtype)

Loaded raw data: 7043 rows, 21 columns
TotalCharges dtype before cleaning: str


Note `TotalCharges` loads as **text** — the data-quality issue we uncovered in Notebook 01 (11 blank values hiding in a column that should be numeric).

## 2. Clean the data with `clean_data()`

**What:** Apply our `clean_data()` function, which replaces the 11 blank `TotalCharges` values with `0` and converts the column to `float`.

**Why:** The blanks all belong to `tenure = 0` customers — brand-new customers who have not been billed yet — so **0 is the correct, defensible fill value** (not a guess or an average). Keeping this in a tested function guarantees the exact same cleaning everywhere it's used.

In [2]:
df = clean_data(df_raw)

# Confirm the column is now numeric and no blanks remain
print("TotalCharges dtype after cleaning:", df["TotalCharges"].dtype)
print("Blank values left:", (df["TotalCharges"].astype(str) == " ").sum())

TotalCharges dtype after cleaning: float64
Blank values left: 0


`TotalCharges` is now a clean **float** column with **zero** blank values — ready for analysis and modelling.

## 3. Check for duplicate rows

**What:** Count exact duplicate customer records.

**Why:** Duplicates would over-weight some customers and leak between train and test sets, quietly inflating model scores.

In [3]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


**No duplicate rows** — every record is a unique customer, so no de-duplication is needed.

## 4. Save the cleaned dataset

**What:** Write the cleaned, still-human-readable data to `data/processed/telco_churn_clean.csv`.

**Why:** This is the **single source of truth** for the rest of the project. The EDA and feature-engineering notebooks load this file instead of re-cleaning the raw data.

Note: we keep the original categorical columns here (no encoding yet) so the EDA charts stay readable. Encoding happens in Notebook 04.

In [4]:
df.to_csv("../data/processed/telco_churn_clean.csv", index=False)
print("Saved -> data/processed/telco_churn_clean.csv")
print(f"Shape: {df.shape}")

Saved -> data/processed/telco_churn_clean.csv
Shape: (7043, 21)


## Section conclusion — what cleaning achieved

- Applied the reusable, unit-tested **`clean_data()`** function from `src/`.
- Converted **`TotalCharges` from text to float** and filled the **11 blanks with 0** (correct for `tenure = 0` new customers).
- Confirmed there are **no duplicate records**.
- Saved a tidy dataset as the single source of truth for every downstream notebook.

**Next:** Notebook 03 uses this clean data to explore *which* kinds of customers churn.